# P2-01: Qwen3 reranking từ gói P1-11
Nhận và giải nén gói handoff trước. Cài repo editable bằng `python -m pip install -e ".[qwen,test]"` trong môi trường notebook; trên Kaggle cần clone/install repo và attach ZIP handoff riêng.
Không cần corpus MMedC, retrieval model hay Qdrant index. Giữ nguyên query/chunk/doc IDs, text và labels; output ở một run mới.
Weak labels và prototype IDs **không phải** nhãn/IDs chính thức. Notebook cũ được giữ trong `notebooks/legacy/`.

In [ ]:
import os
from pathlib import Path
from src.utils.config import load_config
from src.pipeline.handoff import validate_reranking_input
from src.pipeline.reranker_benchmark import benchmark_reranking

# Chỉnh 3 path này hoặc truyền qua environment variables (đường dẫn tuyệt đối trên Kaggle).
HANDOFF_DIR = Path(os.environ.get('P1_HANDOFF_DIR', 'outputs/p1_p2_handoff'))
CONFIG_PATH = Path(os.environ.get('P2_RERANKER_CONFIG', 'configs/reranker/qwen_reranker.yaml'))
OUTPUT_DIR = Path(os.environ.get('P2_OUTPUT_DIR', 'outputs/p2_qwen3_001'))
manifest = validate_reranking_input(HANDOFF_DIR)
{key: manifest[key] for key in ('schema_version', 'query_count', 'candidate_count', 'label_quality', 'id_namespace')}

In [ ]:
config = load_config(CONFIG_PATH)
assert config['reranker']['type'] == 'qwen'
report = benchmark_reranking(HANDOFF_DIR, OUTPUT_DIR, reranker_config=config, ks=[1, 3, 5, 10, 20, 50, 100, 200])
report['after']['macro']

In [ ]:
{'before': report['before']['macro'], 'after': report['after']['macro'],
 'delta_chunks': report['delta_chunks'], 'performance': report['performance'],
 'label_quality': report['label_quality'], 'limitations': report['limitations']}

## Đọc kết quả
`reranked.jsonl`: đủ mọi candidates, thêm `rerank_score`, không threshold/truncate. `config.yaml` snapshot cấu hình Qwen; `input_manifest.json` giữ provenance của gói gốc.
`reranking_benchmark.json`: Recall/Precision/MRR/binary NDCG trước/sau theo query và macro, cùng latency/VRAM. Document Recall@K dùng parent docs của K chunks đầu. Không suy diễn F2 khi chưa có selector hay nhãn thật.
Không sửa config/labels của handoff đã có SHA256; override model/config qua lời gọi trên. Đổi OUTPUT_DIR nếu chạy lại, không ghi đè kết quả cũ.